In [9]:
import pandas as pd
import numpy as np

# 1. Generar el espacio de fechas vectorizado (2024-01-01 al 2027-12-31)
fechas = pd.date_range(start='2024-01-01', end='2027-12-31', freq='D')

df = pd.DataFrame({
    'fecha': fechas,
    'dia': fechas.day,
    'mes': fechas.month,
    'año': fechas.year,
    'dia_semana': fechas.dayofweek, # 0=Lunes, 6=Domingo
    'es_bisiesto': fechas.is_leap_year,
    'evento': 'normal' # Inicialización por defecto
})

# Determinar el último día del mes de forma vectorizada
df['ultimo_dia_mes'] = df['fecha'].dt.days_in_month

# ----------------------------------------------------
# 2. DEFINICIÓN DE FECHAS MÓVILES (Semana Santa, Día de la Madre, Día del Padre)
# ----------------------------------------------------
# Semana Santa mapeada por Jueves, Viernes, Sábado y Domingo
semana_santa = {
    2024: pd.date_range(start='2024-03-28', end='2024-03-31'),
    2025: pd.date_range(start='2025-04-17', end='2025-04-20'),
    2026: pd.date_range(start='2026-04-02', end='2026-04-05'),
    2027: pd.date_range(start='2027-03-25', end='2027-03-28')
}

# Día de la Madre (Segundo Domingo de Mayo) y Día del Padre (Tercer Domingo de Junio)
# Mapeamos el Sábado y Domingo de dichos fines de semana
fines_de_semana_moviles = {
    # Día de la Madre: Sábado y Domingo
    2024: {'madre': ['2024-05-11', '2024-05-12'], 'padre': ['2024-06-15', '2024-06-16']},
    2025: {'madre': ['2025-05-10', '2025-05-11'], 'padre': ['2025-06-14', '2025-06-15']},
    2026: {'madre': ['2026-05-09', '2026-05-10'], 'padre': ['2026-06-20', '2026-06-21']},
    2027: {'madre': ['2027-05-08', '2027-05-09'], 'padre': ['2027-06-19', '2027-06-20']}
}

# ----------------------------------------------------
# 3. APLICACIÓN DE LÓGICAS VECTORIALES Y ASIGNACIÓN DE EVENTOS
# ----------------------------------------------------
# Inicializar máscaras booleanas para optimizar asignación
es_semana_santa = df['fecha'].isin([d for dias in semana_santa.values() for d in dias])
df.loc[es_semana_santa, 'evento'] = 'semana_santa'

for anio, dias in fines_de_semana_moviles.items():
    df.loc[df['fecha'].isin(pd.to_datetime(dias['madre'])), 'evento'] = 'dia_de_la_madre'
    df.loc[df['fecha'].isin(pd.to_datetime(dias['padre'])), 'evento'] = 'dia_del_padre'

# Black Friday (Último viernes de Noviembre)
fechas_black_friday = pd.to_datetime(['2024-11-29', '2025-11-28', '2026-11-27', '2027-11-26'])
df.loc[df['fecha'].isin(fechas_black_friday), 'evento'] = 'black_friday'

# Eventos con Fechas Fijas
df.loc[(df['mes'] == 2) & (df['dia'].isin([13, 14])), 'evento'] = 'san_valentin'
df.loc[(df['mes'] == 7) & (df['dia'].isin([27, 28, 29])), 'evento'] = 'fiestas_patrias'
df.loc[(df['mes'] == 10) & (df['dia'] == 31), 'evento'] = 'halloween'

# Temporada Navideña y Año Nuevo
df.loc[(df['mes'] == 12) & (df['dia'].isin([21, 22, 23])), 'evento'] = 'pre_navidad'
df.loc[(df['mes'] == 12) & (df['dia'] == 24), 'evento'] = 'navidad'
df.loc[(df['mes'] == 12) & (df['dia'] == 25), 'evento'] = 'post_navidad'
df.loc[(df['mes'] == 12) & (df['dia'].isin([28, 29, 30])), 'evento'] = 'pre_nuevo_año'
df.loc[(df['mes'] == 12) & (df['dia'] == 31), 'evento'] = 'nuevo_año'
df.loc[(df['mes'] == 1) & (df['dia'] == 1), 'evento'] = 'post_año_nuevo'

# ----------------------------------------------------
# 4. LÓGICA DE LIQUIDEZ PERUANA (Quincena y Fin de Mes)
# ----------------------------------------------------
# Solo aplicar si el día no ha sido pisado por un evento festivo de mayor jerarquía (ej. Navidad, Fiestas Patrias)
mask_normal = df['evento'] == 'normal'

# Determinar días reales de pago usando operaciones aritméticas sobre vectores
df['dia_pago_quincena'] = np.where(df['dia_semana'] == 5, 14, np.where(df['dia_semana'] == 6, 13, 15))
df['dia_pago_fin_mes'] = np.where(df['dia_semana'] == 5, df['ultimo_dia_mes'] - 1, 
                                  np.where(df['dia_semana'] == 6, df['ultimo_dia_mes'] - 2, df['ultimo_dia_mes']))

# Asignar efectos de liquidez
df.loc[mask_normal & (df['dia'] == df['dia_pago_quincena']), 'evento'] = 'efecto_quincena'
df.loc[mask_normal & (df['dia'] == df['dia_pago_fin_mes']), 'evento'] = 'efecto_fin_mes'

# ----------------------------------------------------
# 5. ESTRUCTURACIÓN DEL OUTPUT FINAL
# ----------------------------------------------------
# Mapeo del campo periodo (YYYYMM)
df['periodo'] = df['fecha'].dt.strftime('%Y%m')

# Limpieza y orden de las columnas requeridas
columnas_finales = ['fecha', 'evento', 'dia', 'mes', 'año', 'periodo']
df_final = df[columnas_finales].copy()

# Visualizar una muestra del dataset limpio
print(df_final['evento'].value_counts())

evento
normal             1275
efecto_quincena      46
efecto_fin_mes       40
semana_santa         16
pre_navidad          12
pre_nuevo_año        12
fiestas_patrias      12
dia_de_la_madre       8
san_valentin          8
dia_del_padre         8
post_año_nuevo        4
halloween             4
black_friday          4
navidad               4
post_navidad          4
nuevo_año             4
Name: count, dtype: int64


In [10]:
df_final1 = df_final[df_final["evento"]!="normal"]

In [ ]:
df_pivot = df_final1.pivot_table(
    index='fecha',
    columns='evento',
    aggfunc='size',
    fill_value=0
).reset_index()

df_pivot

evento,fecha,black_friday,dia_de_la_madre,dia_del_padre,efecto_fin_mes,efecto_quincena,fiestas_patrias,halloween,navidad,nuevo_año,post_año_nuevo,post_navidad,pre_navidad,pre_nuevo_año,san_valentin,semana_santa
0,2024-01-01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,2024-01-15,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,2024-01-31,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,2024-02-13,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,2024-02-14,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,2027-12-25,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
182,2027-12-28,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
183,2027-12-29,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
184,2027-12-30,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [17]:
df_pivot[['fecha', 'black_friday', 'dia_de_la_madre', 'dia_del_padre',
       'efecto_fin_mes', 'efecto_quincena', 'fiestas_patrias', 'halloween',
       'navidad', 'nuevo_año', 'post_año_nuevo', 'post_navidad', 'pre_navidad',
       'pre_nuevo_año', 'san_valentin', 'semana_santa']].to_excel("CalendarFinal.xlsx", index=False)